# Notebook 7: Performance Comparison — NW-flex vs BWA-MEM

This notebook compares NW-flex against BWA-MEM, a widely used
short-read aligner, on simulated reads from real human STR loci. Where
Notebook 6 demonstrated the alignment workflow on a synthetic single locus
with random flanks, here we approach real-world conditions: the loci come
from a panel of human STR sites in hg38, carrying their genuine flank
sequences and repeat motifs. Reads are tiled across haplotypes whose repeat
count differs from the reference, and we ask under what conditions each
method recovers the haplotype's repeat-region length.

**A note on BWA-MEM.** BWA-MEM is a two-stage aligner. It first performs a
*global seed search* of the reference, finding maximal exact matches anywhere 
in the genome. It then performs *local alignment* by Smith-Waterman extension around 
the most promising seeds, scored with an affine-gap scheme. Soft-clipping arises 
in this second stage: when extending past a seed costs more than declaring the 
read end "clipped", BWA-MEM emits an `S` op in the CIGAR. 

## Overview

In this notebook we:

1. **Build the simulation machinery** — locus, haplotype, and read tiling —
   on top of a panel of real STR loci drawn from hg38 (selected to be pure and 
   locally-isolated).

2. **Use a BWA-MEM-compatible scoring scheme.** NW-flex's defaults module
   ships with several alternative scoring schemes; for this notebook we
   select the one whose match, mismatch, gap-open, and gap-extend values
   match BWA-MEM's defaults, so the two methods see the same scoring
   landscape on every read.

3. **Set up three alignment configurations**: BWA-MEM at standard
   parameters; BWA-MEM with the soft-clip penalty raised so high that
   clipping never improves the score; and NW-flex with the STR-aware EP
   pattern from Notebook 4. For BWA-MEM we align both orientations and take
   the better-scoring strand so the comparison does not penalise BWA-MEM
   for orientation artifacts.

4. **Run three validations**, each changing one thing about the simulation:
   - **Length variation only** — haplotype repeat count varies, flanks
     unchanged.
   - **One SNV in the flank** — same sweep with a single base change one bp
     outside the repeat boundary.
   - **Compound repeat** — the locus is two adjacent motifs joined by a
     short interrupting sequence; both counts vary independently.

5. **Tabulate correctness** for each method as a function of read flank
   extent and the haplotype's $\Delta$, and present each result as a
   heatmap with one panel per method.

The headline result is structural: when the only difference between
haplotype and reference is the repeat counts, NW-flex's EP pattern is
guaranteed to find the optimum, while a clip-based aligner is at the mercy
of its scoring tradeoffs at the boundary. Flank variations expose all
aligners to score optimizations that may alter the measured repeat length.

This notebook builds on:

- **Notebook 4**: STR specialization and the phase-preserving EP pattern.
- **Notebook 6**: STR locus simulation and pileup.

The notebook runs on the committed panel TSV plus `bwa` and `samtools` on
`PATH`. If those are missing, the BWA cells skip cleanly with an
installation hint.

## Setup and imports

### Imports

Standard scientific-Python imports plus the core NW-flex pieces this notebook builds on. Notebook-specific helpers (panel loading, BWA wrapper, CIGAR decoder) will be added here as the cells that need them are written.

In [1]:
# 🧙 Notebook magic: autoreload modules
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

# Core NW-flex pieces
from nwflex.dp_core import FlexInput
from nwflex.fast import run_flex_dp_fast
from nwflex.ep_patterns import build_EP_STR_phase
from nwflex.repeats import STRLocus

### Scoring scheme

NW-flex ships several alternative scoring schemes in `nwflex.default`, each
a `(score_matrix, gap_open, gap_extend)` triple keyed by name. The original
NW-flex default uses round-number values chosen for legibility in the
teaching notebooks; the additional schemes target compatibility with
external aligners.

For this notebook we select the **BWA-MEM-compatible** scheme so that
NW-flex and BWA-MEM see matches, mismatches, and gaps under identical
costs. With matched scoring, any disagreement between the two methods
reflects what each algorithm prefers given the same alignment landscape,
not a difference in the landscape itself.

In [2]:
from nwflex.default import get_default_scoring

# BWA-MEM-compatible scoring (+1 / -4 / -6 / -1).
score_matrix, gap_open, gap_extend, a2i = get_default_scoring("bwa_mem")

### BWA-MEM and samtools availability

The BWA arms of this notebook shell out to `bwa` and `samtools`, which must be on `PATH`. The cell below checks both; if either is missing, the BWA cells skip cleanly with an installation hint.

In [3]:
import shutil

required = ("bwa", "samtools")
missing = [t for t in required if shutil.which(t) is None]
if missing:
    print(f"Missing on PATH: {', '.join(missing)}")
    print(f"Install with: conda install -c bioconda {' '.join(missing)}")
    print("BWA-MEM cells will be skipped if these are unavailable.")
else:
    print(f"Found on PATH: {', '.join(required)}")

Missing on PATH: bwa, samtools
Install with: conda install -c bioconda bwa samtools
BWA-MEM cells will be skipped if these are unavailable.


### Input panel

The simulation draws each locus from a panel of human STR sites stored at
`data/hg38_motif_sample_K100.tsv`. The panel was built using [Tandem
Repeat Finder (TRF)](https://tandem.bu.edu/trf/trf.html) run on hg38 and 
the results  filtered to keep only loci that are *pure* (a clean run of a single 
canonical motif) and *locally isolated* (no other repeats in the surrounding 
flank window). Each row gives a genomic location, the motif and tract as found 
in the reference, and the flanking sequences we use to build the simulation reference. 
The panel has 100 example loci for each mono, di and trinucleotide motif.

Schema:

| Column | Meaning |
|---|---|
| `pind` | panel index (locus identifier) |
| `chr`, `start_38`, `stop_38`, `strand` | hg38 coordinates of the repeat tract |
| `type` | canonical motif (e.g. `A`, `AC`, `AGG`) |
| `lflank`, `rflank` | left and right flank sequences |
| `ms_seq` | repeat tract as it appears in hg38 |
| `ref_score_per_base` | per-base purity score (1.0 = perfect tract) |

> *TODO:* a separate appendix notebook (or script) covering the TRF run,
> the purity / isolation filters, and the panel-construction steps would
> make the data reproducible from raw inputs. Out of scope for this
> notebook; link here when written.

In [4]:
from pathlib import Path
import pandas as pd

PANEL_PATH = Path("../data/hg38_motif_sample_K100.tsv")
panel = pd.read_csv(PANEL_PATH, sep="\t")

print(f"Loaded {len(panel):,} loci from {PANEL_PATH}")
print(f"Motif-length breakdown: {panel['type'].str.len().value_counts().sort_index().to_dict()}")
panel.head()

Loaded 6,900 loci from ../data/hg38_motif_sample_K100.tsv
Motif-length breakdown: {1: 400, 2: 1080, 3: 5420}


,pind,chr,start_38,stop_38,strand,type,lflank,rflank,ms_seq,ref_score_per_base
0,0,chr1,36351,36364,+,A,CCATCTCTGGGCCCAGAATGACCCACTGGAGACCTTACAGCTCTCC...,CCCAGCCTGGCGGAAAGAATTTAAATTATAAAAACTTAGAAGTATG...,AAAAAAAAAAAAA,1.0
1,1,chr1,51864,51877,+,A,GTGGGGGTTGAGTTTCACTTTATTTAAAGTGAGTCTTAATCCTCCA...,GAAGATTGATCAGAGAGTACCTCCCCTAAGGGTACATGCAGATAAA...,AAAAAAAAAAAAA,1.0
2,2,chr1,71175,71186,+,A,AGTATATTACTTGGATCCATCTATGTCATTTTCCATGGTTAATGTT...,CCTTAACAAATGATTCTGACAAATATCTTCTCTTTCCAGGGAGAAT...,AAAAAAAAAAA,1.0
3,3,chr1,77174,77195,+,A,CTTCATGTCTAAAACACCGAGAGAGGCACTCTTATGCATTGTTGGT...,GGAAAATAACCAAATGACAATTAGTGAGTACTACTTGCAAAACTTG...,AAAAAAAAAAAAAAAAAAAAA,1.0
4,4,chr1,108545,108561,+,A,TGTACCATGCTCCTCCTTAATCATTCTGAGGTTACATCTTAAGTCC...,GAATGGAGAGAATGCTACATGAGAGAAAGGATCTTATCTATCATGT...,AAAAAAAAAAAAAAAA,1.0


## Simulation setup

We construct the simulation in three stages: locus, haplotype, reads.

### Locus

The locus comes from the panel: a real flank pair, a real repeat motif, and a chosen reference repeat count $N$. Together these give the reference sequence $X = A \cdot R^N \cdot B$. We trim the genomic flanks to a fixed length and a clean motif-edge boundary so the boundary between flank and repeat is unambiguous.

In [5]:
# TODO: load panel TSV, pick a locus, build STRLocus with clean motif-edge flanks

### Haplotype

The haplotype is the sequence we will sample reads from. It uses the same flanks as the reference but a different repeat count $N + \Delta$. The argument that follows lives in how reads of the haplotype line up against the reference.

In [6]:
# TODO: build_haplotype(locus, delta) → haplotype sequence + truth_z_bp

### Reads

The reads are tiled across the haplotype at a fixed read length, constrained to cover at least $K$ bases of each flank. Within that constraint we vary the read start so reads cover different amounts of the left flank — we call this the read's *flank extent*.

In [7]:
# TODO: tile_reads(haplotype, read_len, k_min_flank) and a flank-extent helper

## Three alignment configurations

Each read is aligned against the locus reference under three configurations:

1. **BWA-MEM at standard parameters.**
2. **BWA-MEM with the soft-clip penalty raised** so high that clipping never improves the score.
3. **NW-flex** with the STR-aware EP pattern from Notebook 4.

For NW-flex we use an extended reference with $3N$ repeat copies, so the EP pattern enumerates a comfortable window of expansions relative to $N$. For BWA-MEM we align the read in both orientations and take the better-scoring strand, so the comparison does not penalise BWA-MEM for orientation artifacts.

**Correctness rule.** A read is correct under a method if (a) the method recovers the repeat-region length encoded in the haplotype, and (b) the alignment *spans the repeat*: at least one matched base on the left flank AND at least one matched base on the right flank. We use CIGAR-based length decoders for the BWA arms; for NW-flex the jump structure already gives us the contracted length.

In [8]:
# TODO: align_bwa_both_strands(reference, reads, no_clip=False) wrapper demo

In [9]:
# TODO: align_bwa_both_strands(reference, reads, no_clip=True) wrapper demo

In [10]:
# TODO: NW-flex wrapper using RefAligner over the 3N-extended reference

In [11]:
# TODO: CIGAR-based z_bp decoder + per-arm correctness function

## First validation — length variation only

We run the simulation above with the haplotype flanks left untouched and the repeat count varying over $N + \Delta$ for a small range of $\Delta$. We tabulate correctness for each method as a function of flank extent and $\Delta$ and present the result as a heatmap with one panel per method.

**Expected result.** NW-flex is uniformly correct as long as the read has any flank extent. BWA-MEM at standard parameters fails along the boundary even though the haplotype differs from the reference only in the repeat count. The no-clip arm recovers some of these cases but not all.

In [12]:
# TODO: sweep (flank_extent, Δ) → correctness for each arm

In [13]:
# TODO: 3-panel heatmap (one per arm), x=Δ, y=flank_extent

## Second validation — a single SNV in the flank

We repeat the first validation with one change: the haplotype carries a single SNV at a fixed position. We choose for our example a single base change in the left flank, one base outside the repeat boundary. The locus, the read tiling, and the alignment configurations are unchanged.

**Expected result.** The same heatmap now reads differently. NW-flex remains correct in the regime where it has flank overhang, but not always — sometimes the local sequence and variant have a higher-value alignment. The no-clip arm — the one that recovered the easy cases above — now fails on the reads that cross the SNV.

In [14]:
# TODO: same sweep with one boundary-adjacent SNV injected into the haplotype

In [15]:
# TODO: 3-panel heatmap, same axes as validation 1

## Third validation — compound repeat

The third validation changes the locus structure. The reference is built from two adjacent repeat motifs joined by a short interrupting sequence,

$$X = A \cdot R_1^{N_1} \cdot M \cdot R_2^{N_2} \cdot B,$$

and the haplotype varies both counts independently. The reads, the alignment methods, and the correctness rule carry over.

**Expected result.** The EP pattern for two repeat blocks enumerates the product of allowed counts in a single pass, so NW-flex is correct everywhere on the $(\Delta_1, \Delta_2)$ grid by construction. BWA-MEM is correct on part of the grid; the size and shape of the failure region depends on motif similarity, the length of the interrupting sequence, and the absolute counts.

In [16]:
# TODO: build compound locus + haplotype grid over (Δ₁, Δ₂); align all three arms

In [17]:
# TODO: (Δ₁, Δ₂) heatmap per arm

## Summary

Across all three validations, NW-flex matches the haplotype's repeat-region length whenever the read has flank overhang on both sides. BWA-MEM at standard parameters loses the flank-adjacent reads to soft-clipping; raising the clip penalty recovers the no-variant cases but not the cases with a boundary-adjacent SNV. On compound loci, NW-flex is correct by construction across the full $(\Delta_1, \Delta_2)$ grid while BWA-MEM degrades along a method-dependent failure surface.

The headline is structural: when the only difference between haplotype and reference is repeat count, NW-flex's EP pattern is guaranteed to find the optimum, while a clip-based aligner is at the mercy of its scoring tradeoffs at the boundary.